In [ ]:
!pip install openai
!pip install matplotlib
!pip install seaborn

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = "xxxxxxxx"

In [ ]:
import os
from openai import OpenAI

# OpenAI
client_openai = OpenAI(
    api_key = os.environ['OPENAI_API_KEY'],
)

# all the clients
clients = {
    "gpt-4o-mini" : client_openai,    
}

In [ ]:
for client_name, client in clients.items():
    print(client_name, client)

# Holding a Conversation with the LLM

In [ ]:
import re
import pandas as pd

df = pd.read_csv('Titanic_train.csv')
df

In [ ]:
while True:
    messages = []
    messages.append(
    {
        'role':'user',
        'content':'''
            Here is the schema of my data:
            PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked            
            Note that for Survived, 0 means dead, 1 means alive
            Return the answer in Python code only.
            For your info, I have already loaded the CSV file into a dataframe named df.
        '''
    })

    prompt = input('\nAsk a question: ') 
    if prompt == "quit":
        break
        
    messages.append(
    {
        'role':'user',
        'content':prompt
    })

    for client_name, client in clients.items():
        completion = client.chat.completions.create(
            model =  client_name,
            messages = messages,
            max_tokens = 1024,
            temperature = 0)

        print(f"Code generated by {client_name}")        
        response = completion.choices[0].message.content    
        pattern = re.compile(r'```python\s*([\s\S]*)\n```')
        match = pattern.search(response)
    
        if match:
            extracted_content = match.group(1)
            print('*' * 50, '\n', extracted_content, '\n', '*' * 50)
            if extracted_content.count('\n') > 1:
                exec(extracted_content)   # use this for plotting    
            else:              
                display(eval(extracted_content))  # use this for query        
        else:
            print("No content found within ```python...```.")

**Sample questions**
- What is the proportion of male and female passengers in the dataset? (e.g., Pie chart or Count plot)
- Plot the distribution of passengers by embarkation port
- Plot the survival rate for each passenger class (Pclass)
- How does survival vary for passengers with small vs. large families? (e.g., Grouped Bar chart)
- Can you visualize the survival rate for passengers traveling alone vs. with family?
- Can you visualize the interaction of passenger titles (e.g., Mr., Mrs., Miss, etc.) and survival? (e.g., Count plot or Bar chart)